# Subtitles (.ASS .SRT)
Exploring parsing of .ass and .srt using `pysubs2`.

Note: both of these methods will overwrite the original lyrics structure, timing and even words, which is by design. When user improperly edit it, it will cause lyrics corruption. Both method assumes the user-edited input as a source of file and rebuild the entire lyrics from it, rather than validating/matching from the original.

In [3]:
import json
import pysubs2
from pathlib import Path

## .ass subtitles

In [ ]:
example_json_structure = {
    "start": 0,
    "end": 0,
    "text": "",
    "words": [
        {
            "word": "word",
            "start": 0,
            "end": 0
        }
    ]
}

In [166]:
with open("./subtitles/example.json", encoding="utf-8") as f:
    data = json.load(f)

In [ ]:
def karaoke_symbol(word, time):
    # uses centisecond, convert seconds x100
    return "{" + chr(92) + "k" + str(int(time * 100)) + "}" + word + " "

"""
The following words to karaoke algorithm makes an important assumption.
It assumes within each segment, there are no gaps, these gaps will be filled.
To resolve, must require manual editing in AegisSub.
"""
def words_to_karaoke(words):
    line = ""
    size = len(words)
    for i, word in enumerate(words):
        start = word['start']
        end = word['end']
        next_start = words[i + 1]['start'] if i < size - 1 else end
        line += karaoke_symbol(word['word'], next_start - start)
    return line

words_to_karaoke(data[0]['words'])
# Python print escapes the \ so it shows as \\k100

"{\\k32}Tell {\\k16}me {\\k32}what {\\k32}you {\\k47}want, {\\k28}what {\\k27}you {\\k54}like, {\\k32}it's {\\k85}okay "

In [ ]:
# Writing the files to .ass 
ass = pysubs2.SSAFile()
for text in data:
    event = pysubs2.SSAEvent()
    event.start = pysubs2.make_time(s=text["start"])
    event.end = pysubs2.make_time(s=text["end"])
    event.text = words_to_karaoke(text["words"])
    ass.append(event)
ass.save("subtitles/output_subtitles.ass")

The import flow for .ass files

In [121]:
imported = pysubs2.load('subtitles/output_subtitles.ass')

In [ ]:
def create_segment(start, end, matches):
    """The matches are a list of (duration, text) tuples."""
    words = []
    current = start
    for dur, txt in matches:
        # dur is in centiseconds from the {\k} tag, convert to seconds
        duration = float(dur) / 100.0
        word_start = current
        word_end = current + duration
        words.append({
            "word": txt,
            "start": round(word_start,2),
            "end": round(word_end,2)
        })
        current = word_end
    if words[-1]['end'] < end:
        words[-1]['end'] = end # aegis sub when extending a line, the karaoke word \k doesn't get extended
    return {
        "start": start,
        "end": end,
        "text": "".join(m[1] for m in matches),
        "words": words
    }

In [122]:
import re
parse_pattern = r"\{\\k(\d+)\}([^\{]*)"

In [123]:
edited = []
for line in imported:
    matches = re.findall(parse_pattern, line.text)
    if not matches:
        edited.append(create_segment(line.start/1000, line.end/1000, [("0", line.text)]))
        continue
    edited.append(create_segment(line.start/1000, line.end/1000, matches))

In [ ]:
def detect_and_fix_overlaps(edited):
    """AegisSub can create overlapping segments, there will also be instances where the last word of a segment is extended beyond the end of the segment"""
    for i in range(len(edited) - 1):
        current_line = edited[i]
        next_line = edited[i + 1]
        if current_line["end"] > next_line["start"]:
            print(f"Overlap detected: \n{current_line['text']}\n{next_line['text']}")
            # Adjust the end time of the current line to match the start time of the next line
            current_line["end"] = next_line["start"]
            # fix word timings to fit within the new end time
        for word in current_line["words"]:
            if word["end"] > current_line["end"]:
                print(f"{word['word']} will be adjusted.")
                word["end"] = current_line["end"]
            if word["start"] > current_line["end"]:
                print(f"{word['word']} start after end, might not be shown.")
                word["start"] = current_line["end"]
    return edited

In [ ]:
# detect and fix overlap (this is the process before the detect_and_fix_overlaps function was added)
for i in range(len(edited) - 1):
    current_line = edited[i]
    next_line = edited[i + 1]
    if current_line["end"] > next_line["start"]:
        print(f"Overlap detected: \n{current_line['text']}\n{next_line['text']}")
        # Adjust the end time of the current line to match the start time of the next line
        current_line["end"] = next_line["start"]
        # fix word timings to fit within the new end time
    for word in current_line["words"]:
        if word["end"] > current_line["end"]:
            print(f"{word['word']} will be adjusted.")
            word["end"] = current_line["end"]
        if word["start"] > current_line["end"]:
            print(f"{word['word']} start after end, might not be shown.")
            word["start"] = current_line["end"]

Overlap detected: 
是 你 是 你 梦 见 的 就 是 你
在 哪 里 在 哪 里 见 过 你
你 will be adjusted.
里 will be adjusted.
Overlap detected: 
是 你
是 你
是  will be adjusted.
你 will be adjusted.
你 start after end, might not be shown.
Overlap detected: 
是 你
梦 见 的 就 是 你
你 will be adjusted.
起 will be adjusted.


**.ASS workflow and edge cases handled**

.ASS subtitle should be the preferred method, it creates a more consistent lyrics experience similar to how user would manually edit karaoke lyrics.

Convert each line into

```
{\k123}This {\k456}is {\k789}a {\k012}line
```
- this is supported feature of .ass format to make karaoke timings with AegisSub
Then parsed into when needing to convert back
```
[("123", "This"), ("456", "is"), ("789", "a"), ("012", "line")]
```

**Edge cases**

Line level overlap
```
|-----|
     |------|
```
- currently, the behavior is to **truncate the earlier line** to match the later one
- in addition, all words within that line will be adjusted or likely removed/hidden
- suppose the case 100,This 100,is 100,a 100,line but the actual segment is only 2s long, a line won't even show in AegisSub, user wouldn't notice, so the processing can truncate it safely, producing another consistent result, these word won't show

Extending the line in AegisSub doesn't automatically adjust the last {\k} with it
- if not handled, the last line will still have a short duration
- this is handled in create_segment not the overlap
- it adjust the last word in a segment so it fills the time of each line

Reducing the line duration in AegisSub doesn't automatically adjust the last {\k} with it
- if not handled, the last word in the line will end in a time out of the line, possibly interfering with the next line
- the nested for loop in detect_and_fix_overlaps truncate the last word in the line to match the line duration, it time exceeds it

## .srt flow

In [ ]:
with open("subtitles/example.json", "w", encoding="utf-8") as f:
    json.dump(edited, f, indent=4, ensure_ascii=False)

processing into .srt files with meta tags

In [ ]:
# add 2 lines of short warnings, will be removed by import process
srt = pysubs2.SSAFile()
meta1 = pysubs2.SSAEvent()
meta1.start = pysubs2.make_time(s=0)
meta1.end = pysubs2.make_time(s=0)
meta1.text = "//wx:meta//Warning:"
srt.append(meta1)
meta2 = pysubs2.SSAEvent()
meta2.start = pysubs2.make_time(s=0)
meta2.end = pysubs2.make_time(s=0)
meta2.text = "//wx:meta//DO NOT \nREMOVE //wx:// tags."
srt.append(meta2)

for i, text in enumerate(data): # same as JSON file
    if text['words']: # process the first word in the segment and add //wx:// 
        event = pysubs2.SSAEvent()
        event.start = pysubs2.make_time(s=text['words'][0]['start'])
        event.end = pysubs2.make_time(s=text['words'][0]['end'])
        event.text = f"//wx:{i}//{text['words'][0]['word']}"
        srt.append(event)
    for word in text['words'][1:]: # process the rest of the words in the segment
        event = pysubs2.SSAEvent()
        event.start = pysubs2.make_time(s=word['start'])
        event.end = pysubs2.make_time(s=word['end'])
        event.text = f"{word['word']}"
        srt.append(event)
srt.save("subtitles/output_subtitles.srt")

In [164]:
# parse it back to JSON
imported = pysubs2.load('subtitles/output_subtitles.srt')

In [ ]:
WX_MARKER_RE = re.compile(r"^\s*//wx:(\d+)//\s*", re.IGNORECASE)
segments = []
segment_words = [] # initially
for subtitle in imported:
    if subtitle.text.startswith("//wx:meta//"):
        continue  # Skip meta lines
    match = WX_MARKER_RE.match(subtitle.text)
    if match: # process //wx:// marker
        index = int(match.group(1))
        clean_text = WX_MARKER_RE.sub("", subtitle.text).strip()
        if len(segment_words) > 0:
            # this will run on the second iteration, since initially segment_words is empty
            # if the segment_words is not empty, it means we completed previous segment and now reached the beginning of the next segment, so we need to finalize the previous segment and append it to segments and reset segment_words
            final_words = {
                "text": " ".join([word["word"] for word in segment_words]), "start": round(segment_words[0]["start"], 2), 
                "end": round(segment_words[-1]["end"], 2),
                "words": segment_words
                }
            segments.append(final_words)
            segment_words = []
        # append the text without the marker
        segment_words.append({"word": clean_text, "start": round(subtitle.start / 1000, 2), "end": round(subtitle.end / 1000, 2)})
    else: # process normal words
        clean_text = subtitle.text.strip()
        segment_words.append({"word": clean_text, "start": round(subtitle.start / 1000, 2), "end": round(subtitle.end / 1000, 2)})
# we still have the final line to process
final_words = {
    "text": " ".join([word["word"] for word in segment_words]), "start": round(segment_words[0]["start"], 2), 
    "end": round(segment_words[-1]["end"], 2),
    "words": segment_words
}
segments.append(final_words) # the last one

segments
        

In [ ]:
segments = detect_and_fix_overlaps(segments) # it's more difficult to cause overlap in SubtitleEdit, but still helpful

**.srt workflow and edge cases handled**

Convert every word a separate line in the subtitle, a meta tag of `//wx:{index}//` is applied on new lines.
- the `//` was used so it won't be treated as SDH or other common subtitle meta tags
- as long as the tag doesn't get removed, user can do any operations, such as adding/deleting/merging words

It's more difficult to create overlap in SubtitleEdit, and the default overlap handling shared from .ass is sufficient.

In [159]:
with open("subtitles/Rihanna - Disturbia.json", "w", encoding="utf-8") as f:
    json.dump(segments, f, indent=4, ensure_ascii=False)